<a href="https://colab.research.google.com/github/Eliascc5/English_proficiency_prediction_NLP/blob/main/01_data_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Preprocessing

**Goal:** clean the raw XML-annotated transcripts from the NICT JLE Corpus and
extract each participant's SST proficiency score and their spoken text.

**Pipeline:** parse XML files → extract SST score → strip annotation tags →
remove punctuation → normalize whitespace → save as CSV.

In [ ]:
import re
import os
import csv
from tqdm.notebook import tqdm

In [ ]:
# Mount Google Drive to access the dataset
# Dataset: NICT_JLE_4.1
# Reference: https://alaginrc.nict.go.jp/nict_jle/index_E.html

from google.colab import drive
drive.mount("/content/gdrive")

In [ ]:
# ── Paths ───────────────────────────────────────────────
INPUT_DIR  = '/content/gdrive/MyDrive/NICT_JLE_4.1/LearnerOriginal/'
OUTPUT_DIR = '/content/gdrive/MyDrive/NICT_JLE_4.1/Output/'
OUTPUT_CSV = os.path.join(OUTPUT_DIR, 'preprocessed_corpus.csv')

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── XML tags to strip (content between opening and closing tags) ──
UNWANTED_TAGS = [
    r'<F>.+?</F>',             # Filler / filled pause
    r'<R>.+?</R>',             # Repetition
    r'<OL>.+?</OL>',           # Overlapping speech
    r'<laughter>.+?</laughter>',  # Laughter
    r'<nvs>.+?</nvs>',         # Non-verbal sound
    r'<CO>.+?</CO>',           # (unspecified in original; typically context tag)
    r'<H.+?</H>',              # Personal information
    r'<JP>.+?</JP>',           # Japanese word
    r'<SC>.+?</SC>',           # Self-correction
    r'<SC\?>.+?</SC\?>',     # Unclear self-correction
    r'<\.>.+?</\.>',         # Short pause (2–3 seconds)
    r'<\.\.>.+?</\.\.>',   # Long pause (4+ seconds)
    r'<\?>.+?</\?>',         # Unclear passage
    r'<B>',                    # Candidate speech opening tag
    r'</B>',                   # Candidate speech closing tag
]

# ── Punctuation / special characters to remove ─────────
PUNCT_PATTERN = r'[.?!",;:\(\)-]'

In [ ]:
def extract_score(line):
    """Extract the SST proficiency score (1-9) from an <SST_level> tag."""
    match = re.search(r'<SST_level>(\d)</SST_level>', line)
    if match:
        return int(match.group(1))
    return None


def strip_xml_tags(text):
    """Remove all XML annotation tags from the text."""
    for tag in UNWANTED_TAGS:
        text = re.sub(tag, '', text)
    # Catch any remaining unhandled XML-like tags
    text = re.sub(r'<[^>]+>', '', text)
    return text


def clean_text(text):
    """Apply full cleaning pipeline to a transcript."""
    text = strip_xml_tags(text)
    text = re.sub(PUNCT_PATTERN, '', text)
    text = text.lower()
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def process_file(filepath):
    """
    Process a single raw transcript file.
    Returns (score, cleaned_text) or raises on failure.
    """
    with open(filepath, 'r', encoding='utf8', errors='ignore') as f:
        lines = f.readlines()

    score = None
    transcript_parts = []

    for line in lines:
        if score is None and '<SST_level>' in line:
            score = extract_score(line)
        elif '<B>' in line:
            transcript_parts.append(line)

    if score is None:
        raise ValueError(f"No <SST_level> tag found in {filepath}")

    full_transcript = ' '.join(transcript_parts)
    cleaned = clean_text(full_transcript)

    return score, cleaned

In [ ]:
# ── Collect all .txt files ─────────────────────────────
txt_files = sorted([
    f for f in os.listdir(INPUT_DIR) if f.endswith('.txt')
])

print(f"Found {len(txt_files)} transcript files.")

# ── Process each file ──────────────────────────────────
records = []
errors = []

for filename in tqdm(txt_files, desc='Processing'):
    filepath = os.path.join(INPUT_DIR, filename)
    try:
        score, text = process_file(filepath)
        records.append({'filename': filename, 'score': score, 'transcript': text})
    except Exception as e:
        errors.append((filename, str(e)))

# ── Summary ────────────────────────────────────────────
print(f"\nSuccessfully processed: {len(records)}")
print(f"Errors: {len(errors)}")
if errors:
    print("\nFailed files:")
    for fname, reason in errors:
        print(f"  - {fname}: {reason}")

In [ ]:
# ── Save as single CSV ────────────────────────────────
with open(OUTPUT_CSV, 'w', newline='', encoding='utf8') as f:
    writer = csv.DictWriter(f, fieldnames=['filename', 'score', 'transcript'])
    writer.writeheader()
    writer.writerows(records)

print(f"Corpus saved to: {OUTPUT_CSV}")

# ── Quick sanity check ────────────────────────────────
scores = [r['score'] for r in records]
print(f"\nScore distribution:")
for level in range(1, 10):
    count = scores.count(level)
    bar = '█' * (count // 2)
    print(f"  SST {level}: {count:>4}  {bar}")